In [ ]:
!pip install feedparser

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6104 sha256=76564b03e847f24402ee3343094690664bf2a6b93ec3a6f217ef1a907a80c210
  Stored in directory: c:\users\aft\appdata\local\pip\cache\wheels\3d\4d\ef\37cdccc18d6fd7e0dd7817dcdf9146d4d6789c32a227a28134
Successfully built sgmllib3k

   -------------------- ------------------- 1/2 [feedparser]
   -------------------- ------------------- 1/2 [feedparser]
   ---------------------------------------- 2/2 [feedparser]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install xmltojson


   ---------------------------------------- 2/2 [xmltojson]

Note: you may need to restart the kernel to use updated packages.


In [6]:
import json
import urllib.parse
from datetime import datetime
from pathlib import Path

import requests
import xmltojson


def get_raw_google_news_json(keywords, country):
    """Fetch the raw XML payload from Google News RSS, convert it to JSON, and save it to News/data/raw/."""
    news_root = Path.cwd()
    if news_root.name.lower() != "news" and (news_root / "News").exists():
        news_root = news_root / "News"

    raw_dir = news_root / "data" / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)

    executed_at = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = raw_dir / f"googlenews_raw_{executed_at}.json"

    # 1. Standard Boolean URL-encoding for the search query
    search_query = f"{country} AND ({keywords})"
    encoded_query = urllib.parse.quote(search_query)

    # Core Google News RSS Endpoint URL
    rss_url = f"https://news.google.com/rss/search?q={encoded_query}&hl=en-US&gl=US&ceid=US:en"

    print("==================================================")
    print("📡 Requesting Raw Wire Data from Google News...")
    print(f"🔗 Target: {rss_url}")
    print("==================================================")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) CE49XProjectPipeline"
    }
    response = requests.get(rss_url, headers=headers, timeout=15)

    if response.status_code != 200:
        print(f"❌ Network request failed with Status: {response.status_code}")
        return None

    raw_xml_string = response.text
    raw_json_string = xmltojson.parse(raw_xml_string)
    json_object = json.loads(raw_json_string)

    try:
        raw_items = json_object["rss"]["channel"]["item"]
        print(f"✅ Success! Captured {len(raw_items)} raw, nested JSON entities.")

        print("\n--- SAMPLE RAW NESTED JSON ITEM OBJECT ---")
        print(json.dumps(raw_items[0], indent=4))
        print("------------------------------------------\n")
    except KeyError:
        print("⚠️ Structure mismatch: The query might have returned 0 entries.")

    output_path.write_text(json.dumps(json_object, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"💾 Raw Google News JSON saved: {output_path}")

    return raw_json_string


In [7]:
# --- Search Parameters ---
KEYWORDS = "missile OR explosion OR drone OR strike"
COUNTRY = "Ukraine"

# --- Execute Raw Stream Extraction ---
raw_json_output = get_raw_google_news_json(keywords=KEYWORDS, country=COUNTRY)

📡 Requesting Raw Wire Data from Google News...
🔗 Target: https://news.google.com/rss/search?q=Ukraine%20AND%20%28missile%20OR%20explosion%20OR%20drone%20OR%20strike%29&hl=en-US&gl=US&ceid=US:en
✅ Success! Captured 100 raw, nested JSON entities.

--- SAMPLE RAW NESTED JSON ITEM OBJECT ---
{
    "title": "Ukraine Says It Hit Multiple Targets Inside Russia During Overnight Assault - Radio Free Europe/Radio Liberty",
    "link": "https://news.google.com/rss/articles/CBMikgFBVV95cUxOeGN4ZVk0c2w5dmRRMjJQc3o3dTlmbUhqLVhRTjJ4MnQ0RzltNFQ2bXlaM2lFTFR0UzlndXFYUXp4UkFHYVdYeXY2c19vN0tza3pzZHRNWFpqQTN1YXZTWGJybWIyRlhtY21MUDhJOERtNF9aUG1TdlNEdWtFdl9GUVhRMlVRRGl2cUlMMEY2QlUyQdIBlAFBVV95cUxPYl82clEyNTJpQmc4bjkwS1h1eXJtX0o3QXcyXzZCUXRBSk5zVDFQa2lSX19uWm44MVFiTktVZWNwLWM2cHo2ODBkWDJabHk3Yk1CbF9vSmxvRmQxYXNCQW1yV3V3U3lVRUh6a0l6Sm9PME5vRWhZZV9SeEc1WFhNU1E0ZTJFa3VVbUpNeUZlNlBKVGYy?oc=5",
    "guid": {
        "@isPermaLink": "false",
        "#text": "CBMikgFBVV95cUxOeGN4ZVk0c2w5dmRRMjJQc3o3dTlmbUhqLVhRTjJ4

In [10]:
# --- Process raw Google News JSON -> standardized CSV(s) ---
import json
from pathlib import Path

import pandas as pd

ANALYSIS_KEYWORDS = [
    "missile",
    "explosion",
    "drone",
    "strike",
    "attack",
    "shelling",
    "bombardment",
    "blast",
    "blackout",
    "power outage",
    "infrastructure",
    "energy",
]

RAW_DIR = Path("data") / "raw"
PROC_DIR = Path("data") / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)


def parse_datetime(value):
    if value is None or value == "":
        return pd.NaT
    text = str(value).strip()
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.isna(parsed):
        return pd.NaT
    if not any(token in text.lower() for token in [":", "am", "pm", "t"]):
        parsed = parsed.normalize() + pd.Timedelta(hours=12)
    return parsed


def matches_keywords(text):
    lowered = (text or "").lower()
    return any(keyword in lowered for keyword in ANALYSIS_KEYWORDS)


for raw_file in sorted(RAW_DIR.glob("googlenews_raw_*.json")):
    try:
        obj = json.loads(raw_file.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Failed to read {raw_file}: {exc}")
        continue

    items = obj.get("rss", {}).get("channel", {}).get("item", [])
    if isinstance(items, dict):
        items = [items]

    rows = []
    for item in items:
        title = item.get("title", "")
        link = item.get("link") or item.get("url")
        if not title or not link:
            continue

        source = item.get("source")
        if isinstance(source, dict):
            source = source.get("#text") or source.get("text")
        source = source or "Google News"

        if not matches_keywords(f"{title} {source}"):
            continue

        rows.append(
            {
                "date": parse_datetime(item.get("pubDate") or item.get("publication_date")),
                "news source": source,
                "title": title,
                "link": link,
            }
        )

    if not rows:
        print(f"No matching items found in {raw_file}")
        continue

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["date", "title", "link"])
    df = df.drop_duplicates(subset=["link"]).reset_index(drop=True)
    df = df[["date", "news source", "title", "link"]]

    processed_name = raw_file.name.replace("_raw", "")
    processed_path = PROC_DIR / processed_name.replace(".json", ".csv")
    df.to_csv(processed_path, index=False, encoding="utf-8")
    print(f"Saved processed Google News to: {processed_path} ({len(df)} rows)")


Saved processed Google News to: data\processed\googlenews_20260530_134951.csv (88 rows)
Saved processed Google News to: data\processed\googlenews_20260530_170218.csv (88 rows)


**What this notebook does**
- Fetches Google News RSS results for the selected query and saves the raw XML-to-JSON payload under `data/raw/`.
- Re-processes every saved raw file into a standardized CSV under `data/processed/`.

**Cleaning process**
- Read `rss.channel.item` records from the raw JSON.
- Keep only the shared analysis columns: `date`, `news source`, `title`, `link`.
- Parse the publication date into a full datetime; if the source only gives a date, set the time to 12:00.
- Use the RSS source name when available; otherwise fall back to `Google News`.
- Apply keyword filtering again during processing so the final CSV keeps only relevant items.
- Remove duplicate rows by `link` and save the cleaned CSV with `_raw` removed from the filename.

---
` Merging All News `

In [3]:
# Incremental merge for Google News: keep history in googlenews_all.csv and append newcomers
from pathlib import Path
import pandas as pd

PROC_DIR = Path("data") / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROC_DIR / "googlenews_all.csv"

gn_files = sorted(
    f for f in PROC_DIR.glob("googlenews_*.csv")
    if f.name.lower() != "googlenews_all.csv"
)

frames = []

if out_path.exists():
    try:
        existing_all = pd.read_csv(out_path)
        frames.append(existing_all)
        print(f"Loaded existing master {out_path.name}: {len(existing_all)} rows")
    except Exception as exc:
        print(f"Skipped existing master due to read error: {exc}")

for f in gn_files:
    try:
        df = pd.read_csv(f)
        frames.append(df)
        print(f"Loaded {f.name}: {len(df)} rows")
    except Exception as exc:
        print(f"Skipped {f.name} due to read error: {exc}")

if not frames:
    print(f"No Google News data found in: {PROC_DIR.resolve()}")
else:
    merged = pd.concat(frames, ignore_index=True)

    preferred_cols = ["date", "news source", "title", "link"]
    existing_preferred = [c for c in preferred_cols if c in merged.columns]
    if existing_preferred:
        merged = merged[existing_preferred + [c for c in merged.columns if c not in existing_preferred]]

    if "link" in merged.columns:
        merged = merged.drop_duplicates(subset=["link"], keep="first")
    elif all(c in merged.columns for c in ["title", "date"]):
        merged = merged.drop_duplicates(subset=["title", "date"], keep="first")
    else:
        merged = merged.drop_duplicates()

    merged.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved incremental Google News master: {out_path.resolve()} ({len(merged)} rows)")

merged.head() if 'merged' in locals() else None

Loaded existing master googlenews_all.csv: 108 rows
Loaded googlenews_20260530_134951.csv: 88 rows
Loaded googlenews_20260530_170218.csv: 88 rows
Saved incremental Google News master: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\News\data\processed\googlenews_all.csv (108 rows)


,date,news source,title,link
0,2026-05-30 09:04:05,Al Jazeera,"Ukraine drones strike Russian oil facility, as...",https://news.google.com/rss/articles/CBMitAFBV...
1,2026-05-30 01:34:38,BBC,Ukraine using AI drones to strike vital convoy...,https://news.google.com/rss/articles/CBMiWkFVX...
2,2026-05-30 08:00:00,The New York Times,What to Know About the Drones That Have Been C...,https://news.google.com/rss/articles/CBMijwFBV...
3,2026-05-29 13:23:43,PBS,Russian drone targeting Ukraine crashes into R...,https://news.google.com/rss/articles/CBMiqwFBV...
4,2026-05-30 01:38:00,The Guardian,Ukraine war briefing: Russia preparing ‘massiv...,https://news.google.com/rss/articles/CBMiuAFBV...
